In [1]:
import json
import os
import re

from num2words import num2words

In [2]:
_PUNCT_PATTERN = re.compile(r"[^\w\s]")
_WHITESPACE_PATTERN = re.compile(r"\s+")
_NUMBER_PATTERN = re.compile(r"\d+")


def remove_punctuation(text: str) -> str:
    return _PUNCT_PATTERN.sub(" ", text)


def normalize_numbers(text: str, lang: str = "id") -> str:
    return _NUMBER_PATTERN.sub(lambda m: num2words(int(m.group()), lang=lang), text)


def normalize(text: str) -> str:
    text = remove_punctuation(text)
    text = normalize_numbers(text)
    text = _WHITESPACE_PATTERN.sub(" ", text).strip().lower()
    return text

In [3]:
DATA_PATH = os.path.join("..", "data", "raw", "intent_dataset.jsonl")

records = []
with open(DATA_PATH, encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

print(f"loaded {len(records)} records from {DATA_PATH}")

loaded 5100 records from ../data/raw/intent_dataset.jsonl


In [4]:
for r in records:
    r["text_normalized"] = normalize(r["text"])

changed = [r for r in records if r["text"].strip().lower() != r["text_normalized"]]
print(f"{len(changed)} of {len(records)} rows changed by normalization\n")

for r in changed[:10]:
    print("RAW: ", r["text"])
    print("NORM:", r["text_normalized"])
    print()

4753 of 5100 rows changed by normalization

RAW:  eh, saya mau pesan nasi goreng spesial dua porsi sama ayam bakar satu
NORM: eh saya mau pesan nasi goreng spesial dua porsi sama ayam bakar satu

RAW:  gue mau pesen, um, sate ayam tiga tusuk sama es jeruk dua ya
NORM: gue mau pesen um sate ayam tiga tusuk sama es jeruk dua ya

RAW:  minta mi- mie ayam satu porsi sama bakso dua mangkok
NORM: minta mi mie ayam satu porsi sama bakso dua mangkok

RAW:  anu, saya mau pesan gado-gado satu sama soto ayam dua porsi
NORM: anu saya mau pesan gado gado satu sama soto ayam dua porsi

RAW:  ee, nasi padang satu ya, tapi, em, lauknya rendang sama ayam pop
NORM: ee nasi padang satu ya tapi em lauknya rendang sama ayam pop

RAW:  pesan satu, eh, dua porsi nasi goreng seafood sama satu jus alpukat
NORM: pesan satu eh dua porsi nasi goreng seafood sama satu jus alpukat

RAW:  gue pesen ayam geprek dua porsi sama ca- cap cay satu
NORM: gue pesen ayam geprek dua porsi sama ca cap cay satu

RAW:  saya mau 

In [5]:
OUTPUT_PATH = os.path.join(
    "..", "data", "normalized", "intent_dataset_normalized.jsonl"
)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"wrote {len(records)} normalized records to {OUTPUT_PATH}")

wrote 5100 normalized records to ../data/normalized/intent_dataset_normalized.jsonl
